In [1]:
%%capture
!pip install gensim

**Playlist Dataset**

https://www.cs.cornell.edu/~shuochen/lme/data_page.html


In [2]:
import pandas as pd
from urllib import request

In [3]:
# Get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# Cada linea 3...N representa un playlist
lines = data.read().decode("utf-8").split('\n')[2:]
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

# Ejemplo: 20[\t]Your Love[\t]Nicki Minaj[\n]
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

In [4]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])
print('Songs:')
songs_df.head(5)

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

,title,artist
id,,
0,Gucci Time (w\/ Swizz Beatz),Gucci Mane
1,Aston Martin Music (w\/ Drake & Chrisette Mich...,Rick Ross
2,Get Back Up (w\/ Chris Brown),T.I.
3,Hot Toddy (w\/ Jay-Z & Ester Dean),Usher
4,Whip My Hair,Willow


Word2Vec aprende si dos elementos aparecen en contextos similares, entonces sus vectores(embedding) seran parecidos.

Aplica dos aquitecturas:

*   CBOW (Continuous Bag of Words)
*   Skip-gram




In [5]:
from gensim.models import Word2Vec

# Entrena el algoritmo
model = Word2Vec(
    playlists,
    vector_size=32,  # tamaño del vector(embedding)
    window=20, # mira hasta 20 elementos alrededor de cada canción
    negative=50, # por cada ejemplo real, toma 50 ejemplos falsos
    min_count=1, # incluye incluso canciones que aparecen solo una vez
    workers=4 # usa 4 hilos del CPU para entrenar más rápido
)

In [19]:
import numpy as np

def print_recommendations(p_song_id):
    similar_songs = np.array(
        model.wv.most_similar(positive=str(p_song_id), topn=5)
    )[:,0]
    return  songs_df.iloc[similar_songs]

In [24]:
song_id = 2172

print("Song:")
print(songs_df.iloc[song_id])

# Canciones similares para el id #2172
print("\nEmbedding:")
embeddings = model.wv.most_similar(positive=str(song_id))
print(embeddings)

print("\nSimilarity")
print_recommendations(song_id)

Song:
title     Fade To Black
artist        Metallica
Name: 2172 , dtype: object

Embedding:
[('2849', 0.9988376498222351), ('3167', 0.9971052408218384), ('3094', 0.9956546425819397), ('5586', 0.9949908256530762), ('6624', 0.9947690963745117), ('5479', 0.9944402575492859), ('5549', 0.9943483471870422), ('2704', 0.9936641454696655), ('2904', 0.9936413764953613), ('2640', 0.9934425950050354)]

Similarity


,title,artist
id,,
2849,Run To The Hills,Iron Maiden
3167,Unchained,Van Halen
3094,Breaking The Law,Judas Priest
5586,The Last In Line,Dio
6624,Everybody Wants Some!!!,Van Halen
